# Определение количества дикторов

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import torch
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from pyannote.audio import Pipeline

load_dotenv()

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))

from shared import config

DEVICE = 'cpu'

In [ ]:
HF_TOKEN = os.getenv('HF_TOKEN')

pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-3.1',
    token=HF_TOKEN,
)
pipeline.to(torch.device(f'cpu'))
print('Pipeline загружен.')

## Функция подсчёта дикторов

In [ ]:
def count_speakers(path: str) -> int:
    result = pipeline(path)
    annotation = result.speaker_diarization
    return len(annotation.labels())

# Быстрый тест на одном файле
test_path = str(next(config.GOOD_DIR.glob('*.wav')))
print(f'Тест: {Path(test_path).name} -> {count_speakers(test_path)} диктор(а)')

In [ ]:
df = pd.read_csv(config.DATASET_CSV, encoding='utf-8')
df['_path'] = df.apply(lambda r: str(config.DATA_DIR / r['dir'] / r['filename']), axis=1)

if 'n_speakers' not in df.columns:
    df['n_speakers'] = pd.NA
else:
    df['n_speakers'] = pd.to_numeric(df['n_speakers'], errors='coerce')

need_idx = df[df['n_speakers'].isna() | (df['n_speakers'] == -1)].index.tolist()
print(f'Записей в датасете:  {len(df)}')
print(f'Требуют обработки:   {len(need_idx)}')
print(f'Уже обработано:      {len(df) - len(need_idx)}')

errors = []
for i, idx in enumerate(need_idx):
    path = df.at[idx, '_path']
    try:
        n = count_speakers(path)
    except Exception as e:
        n = -1
        errors.append(f"{df.at[idx, 'filename']}: {e}")
    df.at[idx, 'n_speakers'] = n

    if (i + 1) % SAVE_EVERY == 0:
        df.drop(columns=['_path']).to_csv(config.DATASET_CSV, index=False, encoding='utf-8')
        print(f'[{i+1}/{len(need_idx)}] промежуточное сохранение')
    elif (i + 1) % 10 == 0 or (i + 1) == len(need_idx):
        print(f'{i+1}/{len(need_idx)}  (ошибок: {len(errors)})')

print(f'\Ошибок: {len(errors)}')
for e in errors:
    print(f'{e}')

In [ ]:
df_out = df.drop(columns=['_path'], errors='ignore')
df_out['n_speakers'] = df_out['n_speakers'].astype('Int64')
df_out.to_csv(config.DATASET_CSV, index=False, encoding='utf-8')
print(f'Сохранено: {config.DATASET_CSV}')
print(f'Записей: {len(df_out)}')
df_out[['filename', 'label', 'duration', 'n_speakers']].head(5)

In [ ]:
df = pd.read_csv(config.DATASET_CSV, encoding='utf-8')

valid     = df[df['n_speakers'].notna() & (df['n_speakers'] >= 0)]
errors_df = df[df['n_speakers'] == -1]

print('=' * 50)
print(f'Всего записей:           {len(df)}')
print(f'Успешно обработано:      {len(valid)}')
print(f'Ошибок (n_speakers=-1):  {len(errors_df)}')
print()
print('Распределение количества дикторов:')
for n, cnt in valid['n_speakers'].value_counts().sort_index().items():
    print(f'  {int(n)} диктор(а): {cnt} записей ({cnt/len(valid)*100:.1f}%)')

print()
for label_val, label_str in [(0, 'good'), (1, 'bad')]:
    sub = valid[valid['label'] == label_val]
    print(f'Среднее дикторов [{label_str}]: {sub["n_speakers"].mean():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

max_spk = int(valid['n_speakers'].max())
bins = range(0, max_spk + 2)
for label_val, label_str, color in [(0, 'good', 'steelblue'), (1, 'bad', 'coral')]:
    sub = valid[valid['label'] == label_val]['n_speakers']
    axes[0].hist(sub, bins=bins, alpha=0.7, label=label_str, color=color, align='left')
axes[0].set_xlabel('Количество дикторов')
axes[0].set_ylabel('Количество записей')
axes[0].set_title('Распределение дикторов по классам')
axes[0].legend()
axes[0].set_xticks(list(bins)[:-1])

rows = []
for n in sorted(valid['n_speakers'].unique()):
    sub = valid[valid['n_speakers'] == n]
    rows.append({
        'n_speakers': int(n),
        'pct_bad': (sub['label'] == 1).mean() * 100,
        'count': len(sub),
    })
df_spk = pd.DataFrame(rows)

axes[1].bar(df_spk['n_speakers'].astype(str), df_spk['pct_bad'], color='mediumpurple')
axes[1].set_xlabel('Количество дикторов')
axes[1].set_ylabel('Доля bad, %')
axes[1].set_title('Доля bad по числу дикторов')
for _, r in df_spk.iterrows():
    axes[1].text(str(int(r['n_speakers'])), r['pct_bad'] + 0.5,
                 f"n={int(r['count'])}", ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(exp_dir / 'speaker_diarization_stats.png', dpi=150, bbox_inches='tight')
plt.show()

display(df_spk)